# Introduction

Large reasoning models such as DeepSeek-R1 produce **explicit chain-of-thought traces** wrapped in `<think>` tags before giving a final answer. The [`open-r1/Mixture-of-Thoughts`](https://huggingface.co/datasets/open-r1/Mixture-of-Thoughts) dataset collects thousands of such traces across coding, math, and science problems.

In this lab we will:
1. Load and inspect the dataset.
2. Engineer features that quantify **reasoning depth** (`thought_length`) and **problem difficulty** (`complexity`).
3. Detect **self-correction events** — moments where the model backtracks mid-thought.
4. Visualise the relationship between problem complexity and reasoning length.
5. Extract and study concrete backtracking snippets as qualitative case studies.

---

## 1  Environment Setup

In [ ]:
# Install dependencies (idempotent — safe to rerun)
# uncomment the following lines to install dependencies if you haven't already

#import subprocess, sys
#packages = ["datasets", "pandas", "matplotlib", "seaborn", "scipy"]
#subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet"] + packages)
#print("All packages ready.")

In [ ]:
import re
import warnings

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from datasets import load_dataset

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")
pd.set_option("display.max_colwidth", 120)

SAMPLE_SIZE = 5_000
SELF_CORRECTION_KEYWORDS = ["wait", "actually", "no,", "mistake", "hold on"]
print("Imports OK.")

---

## 2  Data Loading

The dataset ships four configs (`all`, `code`, `math`, `science`). We use `all` and stream the first 5 000 rows to keep memory usage low.

In [ ]:
print(f"Streaming first {SAMPLE_SIZE:,} rows from open-r1/Mixture-of-Thoughts (config=all) …")

ds = load_dataset(
    "open-r1/Mixture-of-Thoughts",
    "all",
    split="train",
    streaming=True,
)

rows = list(ds.take(SAMPLE_SIZE))
print(f"Loaded {len(rows):,} rows.")
print("Raw columns:", list(rows[0].keys()))

In [ ]:
def extract_problem(user_content: str) -> str:
    """Return the problem statement that follows the '# Problem' header."""
    match = re.search(r"#\s*Problem\s*\n+(.+)", user_content, flags=re.DOTALL)
    return match.group(1).strip() if match else user_content.strip()


def extract_thought(assistant_content: str) -> str:
    """Return the text inside <think>…</think>, or empty string if absent."""
    match = re.search(r"<think>(.+?)</think>", assistant_content, flags=re.DOTALL)
    return match.group(1).strip() if match else ""


records = []
for row in rows:
    msgs = row["messages"]
    user_content      = msgs[0]["content"] if len(msgs) > 0 else ""
    assistant_content = msgs[1]["content"] if len(msgs) > 1 else ""

    records.append({
        "source":             row["source"],
        "num_tokens":         row["num_tokens"],
        "problem":            extract_problem(user_content),
        "thought":            extract_thought(assistant_content),
        "assistant_content":  assistant_content,   # kept for snippet extraction
    })

df = pd.DataFrame(records)
print(df.shape)
df.head(3)

In [ ]:
# show an example of a problem and thought
example = df.iloc[0]
print("Problem:\n", example["problem"])
print("\nThought:\n", example["thought"])

---

## Statistical Overview

### 3  Feature Engineering

In [ ]:
# ── thought_length : word count of the reasoning trace ──────────────────────
df["thought_length"] = df["thought"].str.split().str.len().fillna(0).astype(int)

# ── complexity : character length of the problem statement ──────────────────
df["complexity"] = df["problem"].str.len().fillna(0).astype(int)

# ── self_correction : keyword detected inside <think> block ─────────────────
pattern = r"\b(" + "|".join(re.escape(k) for k in SELF_CORRECTION_KEYWORDS) + r")\b"

df["self_correction"] = (
    df["thought"]
    .str.contains(pattern, flags=re.IGNORECASE, regex=True)
    .fillna(False)
)

print(df[["thought_length", "complexity", "self_correction"]].describe())
print(f"\nSelf-correction rate: {df['self_correction'].mean():.1%}")

### 4  Quantitative Analysis

#### 4a  Distribution of thought length and problem complexity

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(df["thought_length"], bins=60, ax=axes[0], color="steelblue")
axes[0].set(title="Distribution of Thought Length", xlabel="Word Count", ylabel="Count")

sns.histplot(df["complexity"], bins=60, ax=axes[1], color="darkorange")
axes[1].set(title="Distribution of Problem Complexity", xlabel="Char Count", ylabel="Count")

plt.tight_layout()
plt.savefig("distributions.png", dpi=120)
plt.show()

**Exercise:** Comment these plots. What are interesting information. How long is 15000 words?

#### 4b  Scatter plot — Thought Length vs. Problem Complexity (coloured by self-correction)

In [ ]:
# Cap extreme outliers for readability (99th percentile)
tl_cap = df["thought_length"].quantile(0.99)
cx_cap = df["complexity"].quantile(0.99)
plot_df = df[(df["thought_length"] <= tl_cap) & (df["complexity"] <= cx_cap)].copy()

fig, ax = plt.subplots(figsize=(10, 6))

colors = {False: "#4079D4", True: "#E07D44"}
labels = {False: "No self-correction", True: "Self-correction detected"}

for flag, group in plot_df.groupby("self_correction"):
    ax.scatter(
        group["complexity"],
        group["thought_length"],
        c=colors[flag],
        label=labels[flag],
        alpha=0.35,
        s=15,
        edgecolors="none",
    )

# Trend line
m, b, r, p, _ = stats.linregress(plot_df["complexity"], plot_df["thought_length"])
x_line = sorted(plot_df["complexity"])
ax.plot(x_line, [m * x + b for x in x_line], color="crimson", linewidth=1.5,
        label=f"Trend (r={r:.3f}, p={p:.2e})")

ax.set(title="Thought Length vs. Problem Complexity",
       xlabel="Problem Complexity (char count)",
       ylabel="Thought Length (word count)")
ax.legend()
plt.tight_layout()
plt.savefig("scatter_thought_vs_complexity.png", dpi=120)
plt.show()

print(f"\nPearson r = {r:.4f}  |  p-value = {p:.4e}")

**Questions:** Is the linear curve really a good fit? Is the "complexity" a well defined quatity?

#### 4c  Thought length by self-correction flag

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.boxplot(
    data=df[df["thought_length"] <= tl_cap],
    x="self_correction",
    y="thought_length",
    palette=["#4C72B0", "#DD8452"],
    ax=ax,
)
ax.set(title="Thought Length by Self-Correction Flag",
       xticklabels=["No self-correction", "Self-correction"],
       xlabel="", ylabel="Word Count")
plt.tight_layout()
plt.savefig("boxplot_self_correction.png", dpi=120)
plt.show()

# Mann-Whitney U test
u_stat, u_p = stats.mannwhitneyu(
    df.loc[df["self_correction"],  "thought_length"],
    df.loc[~df["self_correction"], "thought_length"],
    alternative="greater",
)
print(f"Mann-Whitney U = {u_stat:.0f}, p = {u_p:.4e}")
print("Interpretation: are self-correcting traces significantly longer?")

**Exercise:** Find and display one of the longest reasonings without self-correcting words. Is there anything noticable?

---

## Case Studies of Self-Correction

### 5  Qualitative Extraction — Backtracking Snippets

A **backtracking snippet** captures 200 characters of context *before* a self-correction keyword and 400 characters *after* it, letting us observe exactly how the model recovers from a reasoning error.

In [ ]:
def extract_backtracking_snippet(
    thought: str,
    keywords: list[str] = SELF_CORRECTION_KEYWORDS,
    pre: int = 200,
    post: int = 400,
) -> dict | None:
    """
    Search *thought* for the first occurrence of any self-correction keyword.

    Returns a dict with:
        keyword   – the matched word/phrase
        position  – character index in the thought string
        snippet   – pre + keyword + post context
    or None if no keyword is found.
    """
    pattern = r"\b(" + "|".join(re.escape(k) for k in keywords) + r")\b"
    match = re.search(pattern, thought, flags=re.IGNORECASE)
    if match is None:
        return None

    start = match.start()
    end   = match.end()
    snippet = thought[max(0, start - pre) : end + post]

    # Mark the keyword for readability
    kw_in_snippet = thought[start:end]
    snippet = snippet.replace(kw_in_snippet, f">>>[ {kw_in_snippet} ]<<<", 1)

    return {"keyword": kw_in_snippet, "position": start, "snippet": snippet}


# Quick sanity check
test_thought = "Hmm, if I double the value I get 10. wait, that is wrong. Let me reconsider."
result = extract_backtracking_snippet(test_thought)
print("Keyword found:", result["keyword"])
print(result["snippet"])

In [ ]:
def print_case_studies(
    df: pd.DataFrame,
    n: int = 5,
    min_thought_words: int = 200,
) -> None:
    """
    Print *n* backtracking case studies drawn from rows where self-correction
    was detected and the thought trace is at least *min_thought_words* words long.
    """
    pool = df[
        df["self_correction"] & (df["thought_length"] >= min_thought_words)
    ].sample(frac=1, random_state=42).head(n)

    for i, (_, row) in enumerate(pool.iterrows(), 1):
        info = extract_backtracking_snippet(row["thought"])
        if info is None:
            continue

        sep = "-" * 72
        print(f"\n{'='*72}")
        print(f"Case Study {i}  |  source: {row['source']}")
        print(f"Thought length: {row['thought_length']:,} words  "
              f"|  Problem complexity: {row['complexity']:,} chars")
        print(f"Self-correction keyword: '{info['keyword']}'  "
              f"at character {info['position']:,}")
        print(sep)
        print("PROBLEM (first 300 chars):")
        print(row["problem"][:300], "…")
        print(sep)
        print("BACKTRACKING SNIPPET:")
        print(info["snippet"])
        print("="*72)

In [ ]:
print_case_studies(df, n=5)

### 6  Keyword Frequency Breakdown

Which self-correction keywords appear most often?

In [ ]:
kw_counts = {}
for kw in SELF_CORRECTION_KEYWORDS:
    p = r"\b" + re.escape(kw) + r"\b"
    kw_counts[kw] = df["thought"].str.count(p, flags=re.IGNORECASE).sum()

kw_df = pd.Series(kw_counts, name="occurrences").sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(x=kw_df.index, y=kw_df.values, palette="viridis", ax=ax)
ax.set(title="Self-Correction Keyword Frequency",
       xlabel="Keyword", ylabel="Total Occurrences")
for p_bar in ax.patches:
    ax.annotate(f"{int(p_bar.get_height()):,}",
                (p_bar.get_x() + p_bar.get_width() / 2, p_bar.get_height()),
                ha="center", va="bottom", fontsize=10)
plt.tight_layout()
plt.savefig("keyword_frequency.png", dpi=120)
plt.show()

print(kw_df.to_string())

**Exercise:** add more words in the list of self-correcting words and recompute the barplot above. What are other words used for iterating in the reasoning?

---

## 7  Summary & Takeaways

| Finding | Insight |
|---|---|
| **Thought length ↔ problem complexity** | Positive correlation: harder (longer) problems elicit longer reasoning traces. |
| **Self-correction rate** | A substantial fraction of traces contain backtracking keywords, confirming that the model frequently revises its reasoning mid-stream. |
| **Correcting traces are longer** | The Mann-Whitney test checks whether self-correcting traces are statistically significantly longer — consistent with the model spending more tokens to recover from errors. |
| **Dominant keyword** | *"actually"* and *"wait"* tend to be the most frequent signals of backtracking. |

> **Pedagogical note:** Self-correction is a hallmark of deliberate, System-2-style reasoning. Datasets like Mixture-of-Thoughts are valuable precisely because they expose this inner dialogue, enabling fine-tuning and analysis of *how* models reason, not just *what* they answer.

Do you agree with these conclusions?